# 01 — Baseline VLM (MedGemma)

**Baseline « par prompting »** demandée par l'appel d'offre : on prompte le modèle multimodal pré-entraîné **Google MedGemma-4b** (réf. R4) avec `prompts/baseline_prompt.txt`, sur les radiographies thoraciques frontales. Sortie attendue : JSON du contrat (`normal` / `suspected_opacity` / `uncertain`).

> **Position non clinique.** Prototype pédagogique. Aucune sortie ne doit servir au diagnostic. Un score élevé sur ce jeu synthétique ne constitue pas une performance médicale.

**Prérequis** : avoir accepté la licence sur https://huggingface.co/google/medgemma-4b-it et s'être authentifié (`hf auth login`). Le modèle (~8 Go) est téléchargé au premier appel.

In [ ]:
from pathlib import Path
import sys, json
sys.path.append(str(Path('..').resolve()))

from src.medgemma_baseline import medgemma_predict, load_prompt
from src.guardrails import apply_safety_guardrails

print(load_prompt('baseline')[:300], '...')

## Une image — sortie JSON + garde-fous
Charge une image, applique le prompt baseline, puis les garde-fous (validation du schéma, warning obligatoire, bascule vers `uncertain` si besoin).

In [ ]:
sample = Path('../data/sample_images/CXR_SYN_002_suspected_opacity.png')
pred = apply_safety_guardrails(medgemma_predict(sample, mode='baseline'))
print(json.dumps(pred, indent=2, ensure_ascii=False))

## Les 30 cas — baseline
Boucle sur `data/synthetic_cases.csv`, collecte les prédictions et calcule les métriques minimales (accuracy, macro-F1, taux JSON valide, taux warning, taux d'incertitude).

In [ ]:
import csv
from src.guardrails import validate_prediction
from src.metrics import summarize_metrics, confusion_counts

ROOT = Path('..').resolve()
cases = list(csv.DictReader(open(ROOT / 'data' / 'synthetic_cases.csv')))

def run_all(mode):
    rows = []
    for c in cases:
        p = apply_safety_guardrails(medgemma_predict(ROOT / c['image_path'], mode=mode))
        valid, _ = validate_prediction(p)
        rows.append({'case_id': c['case_id'], 'label': c['label'],
                     'predicted_class': p['predicted_class'], 'confidence': p['confidence'],
                     'json_valid': valid, 'warning': p.get('warning',''),
                     'latency_ms': p.get('latency_ms', 0)})
    return rows

baseline_rows = run_all('baseline')
print(json.dumps(summarize_metrics(baseline_rows), indent=2))
print('confusion (true__pred):', confusion_counts([r['label'] for r in baseline_rows],
                                                   [r['predicted_class'] for r in baseline_rows]))

## Comparaison baseline vs prompt amélioré
Le prompt amélioré (`prompts/improved_prompt.txt`) ajoute une règle d'incertitude stricte (confiance < 0.60 → `uncertain`). On compare les métriques avant/après : c'est l'« amélioration mesurée » (20 % de la note).

In [ ]:
improved_rows = run_all('improved')
before = summarize_metrics(baseline_rows)
after = summarize_metrics(improved_rows)
print(f"{'metric':18}{'baseline':>12}{'improved':>12}")
for k in before:
    print(f"{k:18}{str(before[k]):>12}{str(after[k]):>12}")

## Limites
- Jeu **synthétique** : valide la chaîne logicielle, pas une performance clinique.
- MedGemma reste un modèle généraliste médical : conditions d'usage et licence à documenter (R4).
- Les hallucinations textuelles (code HT) doivent être repérées à la main dans les justifications.
- Voir `eval/run_evaluation.py --model medgemma` pour reproduire ce run hors notebook, et `eval/build_error_register.py` pour le registre d'erreurs.